<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L05-remaining-useful-life/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L05-remaining-useful-life/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/predictive-maintenance/lessons/P03-L05-remaining-useful-life/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/predictive-maintenance/lessons/P03-L05-remaining-useful-life/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P03-L05 · Remaining useful life, and its error bars

**You will build:** a remaining-useful-life model that answers with a *distribution* and not
a number. An exponential degradation fit with a fleet prior, a posterior over the two
parameters that tightens as evidence arrives, a Monte-Carlo predictive interval for the
hour the unit crosses its failure level, and four ways of scoring the result — the RMSE
everybody reports, the asymmetric prognostic score from the C-MAPSS challenge, the
alpha-lambda accuracy cone, and the warning lead time your maintenance contract actually
cares about.

**Time:** ~80 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (the profiler and the tier gate),
`P03-L01-alarm-economics` (the lead-time contract and the habit of pricing a decision),
`P03-L04-labelling-run-to-failure` (the run table — `Run(unit, end_hour, kind)`, failures,
suspensions and censored units — arrives here already built; this module never parses a
work order).

By the end you will be able to:

1. Implement log_signal and fit_posterior so an exponential degradation model becomes a
   Gaussian posterior over an intercept and a slope, and measure that posterior
   tightening as readings arrive.
2. Implement rul_samples and rul_track to turn that posterior into a predictive RUL
   distribution re-computed at every checkpoint, and measure how far its interval
   narrows between a third of life and the last fifth.
3. Implement rul_rmse, phm_score and cone_breakdown, and explain why two of those three
   cannot tell an early error from a late one.
4. Implement warning_lead and measure, in units rather than in hours, how often each
   model gives less warning than the maintenance contract requires.
5. Implement best_quantile, measure which quantile of one predictive distribution each
   metric selects when all four are run on the same rows, and demonstrate from those
   numbers that a later model can score a lower RMSE than a cautious one while being
   late an order of magnitude more often.

**The data is synthetic and the generator is in this notebook.** Nothing is downloaded and
nothing needs a network. `MODULES.md` names NASA's C-MAPSS turbofan sets for this module;
that archive sits behind a repository page, so it cannot be on a path the grader needs
(gate 11). It is cited in `claims.yaml` with its licence line and direct URL as an optional
extension you can run this same code against afterwards. The method transfers. The
particular numbers describe no real fleet.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import sys
import time
import traceback
from typing import Callable, NamedTuple, Sequence

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The fleet. 100 archived units whose histories you learn the prior from, 80 units in
# service whose remaining life you are asked for, one health-index reading per hour.
N_HISTORY = 100
N_FLEET = 80
N_HOURS = 1100
SEED = 20260922

# The health index from module 3, as module 4 handed it over: about 1.0 on a well unit and
# rising as a defect develops. HEALTHY is that baseline; FAILURE_LEVEL is the value module 4's
# labelling policy calls a failure. It is a definition written into the labelling policy, not
# a quantity you estimate, which is why the lesson treats it as known.
HEALTHY = 1.0
FAILURE_LEVEL = 9.0

# The run outcomes, unchanged from module 4. A failure gives you an end-of-life; a suspension
# and a censored unit give you a degradation slope and no end-of-life at all.
FAILURE = "failure"
SUSPENSION = "suspension"
CENSORED = "censored"
RUN_KINDS = (FAILURE, SUSPENSION, CENSORED)

# The observation plan. You read the health index every SAMPLE_EVERY hours when you fit, and
# you re-run the prognostic every CHECK_EVERY hours from FIRST_CHECK onwards.
SAMPLE_EVERY = 8
FIRST_CHECK = 96
CHECK_EVERY = 24

# The maintenance contract, carried forward from module 1. LEAD_HOURS is the notice the
# workshop needs; ACTION_RUL is the predicted remaining life at which you raise the job.
LEAD_HOURS = 48.0
ACTION_RUL = 72.0

# Evaluation. EVAL_HORIZON bounds the region in which a prognosis is worth scoring at all,
# ALPHA is the half-width of the alpha-lambda accuracy cone as a fraction of true RUL, and
# RUL_CAP is the longest remaining life this notebook will ever report.
EVAL_HORIZON = 300.0
ALPHA = 0.20
RUL_CAP = 1500.0

# The two time constants of the C-MAPSS asymmetric score. The late one is smaller, so the
# same error in hours costs more when the prognosis arrives late. Section 8 prints the ratio.
LATE_TAU = 10.0
EARLY_TAU = 13.0

# The quantiles of the predictive RUL distribution you will sweep in section 11. Each one is
# a different model: same distribution, different answer to "which number do I act on?".
QUANTILES = np.round(np.arange(0.05, 0.951, 0.05), 2)
N_SAMPLES = 1500

# Below this the log transform in exercise 1 would take the log of a non-positive number.
LOG_FLOOR = 1e-3

# The failure level on the log scale, computed from the two constants above rather than
# typed, because every predictive interval in this notebook is a crossing time of this one
# number. It is a definition from module 4's labelling policy; nothing here estimates it.
Y_FAIL = float(np.log(FAILURE_LEVEL - HEALTHY))


class Run(NamedTuple):
    """One unit's observation window. Module 4 built this table; this module consumes it."""
    unit: int
    end_hour: int      # the hour the run stopped being observed
    kind: str          # one of RUN_KINDS


class Posterior(NamedTuple):
    """A Gaussian posterior over the two parameters of the log-degradation line."""
    mean: np.ndarray   # shape (2,): [intercept, slope]
    cov: np.ndarray    # shape (2, 2)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("log_signal",),
    "exercise 2": ("fit_posterior",),
    "exercise 3": ("rul_samples",),
    "exercise 4": ("rul_track",),
    "exercise 5": ("rul_rmse",),
    "exercise 6": ("phm_score",),
    "exercise 7": ("cone_breakdown",),
    "exercise 8": ("warning_lead",),
    "exercise 9": ("best_quantile",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (rul_samples)"; several -> "exercises 1, 2 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. A fleet that degrades exponentially, and the two things you never see

A bearing does not wear linearly. Damage accelerates damage: a spall spreads, the spread
raises the load on what is left, and the health index bends upwards. Write that as

> `health(t) = HEALTHY + exp(a + b*t + noise)`

and the whole model becomes a straight line in `log(health - HEALTHY)`. That is the
exponential degradation model, and its two parameters are what you are estimating: `a` is
how damaged the unit was when you started watching, `b` is how fast it is getting worse.

Read the generator. Three things in it decide everything downstream:

- **A unit fails when the noise-free curve reaches `FAILURE_LEVEL`.** That hour is the
  ground truth. Your model never sees it; it sees a noisy health index and has to find it.
- **A suspended or censored unit has a slope and no end-of-life.** Module 4's point, one
  layer up: censoring destroys the failure time, not the degradation rate, so section 3
  uses every archived unit for the prior and only the failures for scoring.
- **After the work the unit comes back as a new one.** Every hour at or after `end_hour`
  belongs to a different machine. Fit on it and your slope collapses.

In [ ]:
def generate_fleet(seed: int = SEED, n_history: int = N_HISTORY, n_fleet: int = N_FLEET,
                   n_hours: int = N_HOURS) -> tuple[np.ndarray, list[Run]]:
    """Deterministic synthetic run-to-failure fleet: health index plus module 4's run table.

    Returns `(health, runs)`:
      * `health` has shape (n_history + n_fleet, n_hours). About 1.0 on a well unit, rising
        exponentially as a defect develops, and reset after the unit is overhauled.
      * `runs` is one `Run` per unit, in unit order: units 0 .. n_history-1 are the archive
        you fit the prior from, the rest are the fleet in service you are asked about.

    Same seed, same fleet, on any machine. Nothing is downloaded.

    Module 6 takes this generator unchanged; it is written to be lifted.
    """
    rng = np.random.default_rng(seed)
    n = n_history + n_fleet
    t = np.arange(n_hours, dtype=float)

    # Each unit: an initial damage level `a` and a life `life` hours, which together fix the
    # slope, because the noise-free curve must reach Y_FAIL exactly at `life`.
    a = rng.normal(-2.5, 0.45, n)
    life = rng.integers(280, 1250, n)
    b = (Y_FAIL - a) / life
    y = a[:, None] + b[:, None] * t[None, :] + rng.normal(0.0, 0.22, (n, n_hours))

    # One unit in five is pulled for something that is not a failure — module 4's suspension.
    suspended = set(int(u) for u in rng.permutation(n)[: n // 5])

    runs: list[Run] = []
    for u in range(n):
        if u in suspended:
            end = int(rng.integers(int(0.35 * life[u]), int(0.85 * life[u])))
            kind = SUSPENSION
        elif life[u] >= n_hours:
            end = n_hours                      # still running when the history ends
            kind = CENSORED
        else:
            end = int(life[u])
            kind = FAILURE
        if end < n_hours:
            # Overhauled and back in service: a fresh unit, a fresh damage level, a fresh
            # slope. These hours are data about a machine that no longer exists.
            a2 = rng.normal(-2.5, 0.45)
            b2 = (Y_FAIL - a2) / int(rng.integers(280, 1250))
            tail = np.arange(n_hours - end, dtype=float)
            y[u, end:] = a2 + b2 * tail + rng.normal(0.0, 0.22, n_hours - end)
        runs.append(Run(u, end, kind))

    return HEALTHY + np.exp(y), runs


HEALTH, RUNS = generate_fleet()
HISTORY = RUNS[:N_HISTORY]
FLEET = RUNS[N_HISTORY:]
FLEET_FAILURES = [r for r in FLEET if r.kind == FAILURE]

print(f"health index    {HEALTH.shape} float64, {HEALTH.nbytes / 2**20:.1f} MiB")
print(f"failure level   health {FAILURE_LEVEL:.1f} = log({FAILURE_LEVEL - HEALTHY:.1f}) = "
      f"{Y_FAIL:.4f} on the log scale")
for label, table in (("archive", HISTORY), ("in service", FLEET)):
    counts = {k: sum(1 for r in table if r.kind == k) for k in RUN_KINDS}
    print(f"{label:<15} {len(table)} units · " +
          " · ".join(f"{v} {k}" for k, v in counts.items()))
print(f"scored later    {len(FLEET_FAILURES)} fleet units reached a failure inside the window; "
      f"the rest have no end-of-life to score against")

## 2. Exercise 1 — `log_signal()`

Everything downstream works on `y = log(health - HEALTHY)`, where the curve is a straight
line and the two parameters are an intercept and a slope. Two details decide whether this
is safe.

The subtraction matters: `log(health)` is not linear in `t`, `log(health - HEALTHY)` is.
Leave the baseline in and you are fitting a line to something that is not one, and the
slope you get will depend on where in life you happened to fit it.

The floor matters too. A healthy unit's index wanders either side of 1.0, so
`health - HEALTHY` goes negative on perfectly ordinary rows, and `log` of that is a
`nan` that will propagate silently through a least-squares fit to a posterior mean of
`nan`. Clamp at `LOG_FLOOR` and the row stays finite and very negative, which is the truth:
the unit is as healthy as this index can express.

<details><summary>💡 Hint 1 — what to think about</summary>

Two separate things can break the straight line. Which quantity is exponential in time:
the health index itself, or its distance above the healthy baseline? And a healthy unit's
reading sits a hair below that baseline on ordinary rows: what does a logarithm of a
negative number give you in numpy? It does not raise, and that is the trap.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Turn the input into a float array first, so a scalar and a 2-D block of the fleet go down
the same path and keep their shape. Subtract the baseline, raise anything below the floor
up to the floor, and only then take the natural log. The order is the whole exercise: a
floor applied after the log arrives too late, because the nan already exists.
</details>

In [ ]:
def log_signal(health: np.ndarray | float) -> np.ndarray:
    """Put the health index on the log scale the degradation model is linear in.

    Subtract the healthy baseline `HEALTHY`, clamp the result up to `LOG_FLOOR` so a healthy
    unit's noise cannot produce a nan, and take the natural log. Shape is preserved, and the
    result is always float, whatever came in.

    Returns: an `np.ndarray` of float64 with the same shape as `health`.

    Example:
        >>> float(log_signal(1.0 + np.e))          # exp(1) above the baseline
        1.0
        >>> float(log_signal(0.2))                 # below the baseline: floored, never nan
        -6.907755278982137
        >>> log_signal(np.array([[1.0 + np.e, 1.0 + np.e**2]])).shape
        (1, 2)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_log_signal() -> None:
    one = float(log_signal(HEALTHY + np.e))
    assert abs(one - 1.0) < 1e-12, (
        f"log_signal(HEALTHY + e) should be exactly 1.0, got {one:.6f}. A value of "
        f"{np.log(HEALTHY + np.e):.6f} means you took log(health) without subtracting the "
        "healthy baseline first — that curve is not a straight line in t"
    )
    floored = float(log_signal(0.2))
    assert abs(floored - np.log(LOG_FLOOR)) < 1e-12, (
        f"a health index of 0.2 is below the baseline; log_signal must floor it at "
        f"log(LOG_FLOOR) = {np.log(LOG_FLOOR):.4f} and got {floored}. A nan here poisons every "
        "least-squares fit downstream and never raises"
    )
    assert np.isfinite(log_signal(np.array([0.5, 1.0, 1.0 + np.e]))).all(), (
        "a healthy unit's index wanders either side of 1.0, so log_signal must be finite on "
        "every row of real data — clamp before the log, not after"
    )
    shaped = log_signal(HEALTH[:3, :5])
    assert shaped.shape == (3, 5) and shaped.dtype == np.float64, (
        f"shape and dtype must survive: expected (3, 5) float64, got {shaped.shape} "
        f"{shaped.dtype}. Use np.asarray and numpy's own log rather than a Python loop"
    )
    print(f"exercise 1 looks right — the fleet's log signal runs from "
          f"{log_signal(HEALTH).min():.2f} to {log_signal(HEALTH).max():.2f}")

In [ ]:
_try("exercise 1", _check_log_signal)

## 3. The fleet prior — what the archive knows before this unit says anything

At hour 96 a unit has given you twelve readings of a noisy index. Fit a line to those alone
and the slope is worth very little. The archive is worth more: a hundred units have already
done this, and the spread of their intercepts and slopes is a perfectly good prior.

Note which units go into it. **Every** archived unit contributes, including the suspended
and censored ones. Module 4 spent a whole lesson on why a suspension is not a negative
example; here is the other half of that idea. A suspension destroys the *failure time* and
leaves the *degradation rate* intact, so a unit pulled for a line reconfiguration still
tells you how fast units of this type go bad. Throwing those rows away because "they never
failed" discards a fifth of the fleet's information about `b`.

This function is given. Read it, run it, and look at what it prints: the prior slope, its
spread, and the residual standard deviation that becomes `noise_sd` in exercise 2.

In [ ]:
def fleet_prior(health: np.ndarray, runs: Sequence[Run],
                sample_every: int = SAMPLE_EVERY) -> tuple[np.ndarray, np.ndarray, float]:
    """Empirical prior over (intercept, slope) from the archived units. Given to you.

    One ordinary least-squares line per archived unit, fitted only on hours strictly before
    that unit's `end_hour`, then the mean and covariance of those hundred parameter pairs.
    `noise_sd` is the pooled residual standard deviation, which is what exercise 2 needs.

    Returns `(prior_mean, prior_cov, noise_sd)`.
    """
    rows, residuals = [], []
    for run in runs:
        hours = np.arange(0, run.end_hour, sample_every, dtype=float)
        if hours.size < 4:
            continue
        y = log_signal(health[run.unit, hours.astype(int)])
        design = np.column_stack([np.ones_like(hours), hours])
        coef, *_ = np.linalg.lstsq(design, y, rcond=None)
        rows.append(coef)
        residuals.append(y - design @ coef)
    params = np.asarray(rows)
    return params.mean(axis=0), np.cov(params.T), float(np.std(np.concatenate(residuals)))


_PRIOR: list = []


def prior() -> tuple[np.ndarray, np.ndarray, float]:
    """The fleet prior, computed once and reused. Depends on YOUR log_signal."""
    if not _PRIOR:
        _PRIOR.append(fleet_prior(HEALTH, HISTORY))
    return _PRIOR[0]


def _show_prior() -> None:
    prior_mean, prior_cov, noise_sd = prior()
    sd = np.sqrt(np.diag(prior_cov))
    print(f"prior intercept  {prior_mean[0]:+.3f} ± {sd[0]:.3f}")
    print(f"prior slope      {prior_mean[1]:+.5f} ± {sd[1]:.5f} per hour")
    print(f"noise sd         {noise_sd:.3f} on the log scale")
    typical = (Y_FAIL - prior_mean[0]) / prior_mean[1]
    fast = (Y_FAIL - prior_mean[0]) / (prior_mean[1] + sd[1])
    print(f"a unit at the prior mean reaches the failure level at hour {typical:,.0f}; one a "
          f"single\nstandard deviation faster reaches it at hour {fast:,.0f}. That is the "
          f"spread you start from,\nand it is {typical - fast:,.0f} hours wide before this "
          f"unit has said anything at all.")
    kinds = {k: sum(1 for r in HISTORY if r.kind == k) for k in RUN_KINDS}
    print(f"built from {len(HISTORY)} archived units: " +
          ", ".join(f"{v} {k}" for k, v in kinds.items()) +
          " — the ones that never failed still have a slope")


_try("the fleet prior", _show_prior, needs=("exercise 1",))

## 4. Exercise 2 — `fit_posterior()`

Bayesian linear regression, the whole of it, in four lines of numpy. With a design matrix
`X = [1, t]`, a Gaussian prior `N(m0, S0)` on the two parameters and known noise variance
`s^2`, the posterior is Gaussian with

> `S = inv(inv(S0) + X.T @ X / s**2)` and `m = S @ (inv(S0) @ m0 + X.T @ y / s**2)`

Two properties are worth noticing while you write it, because both are graded. With no
data at all the answer is the prior, exactly. And `S` shrinks monotonically as rows arrive —
that shrinking is the only reason the predictive interval in section 6 narrows.

<details><summary>💡 Hint 1 — what to think about</summary>

Work out what the formula becomes in the two corners the check tries. With no readings at
all the data terms vanish: what is left? With a very tight prior, one reading should
barely move it. Then look at the formula again: it divides by the noise VARIANCE, and the
argument you are handed is a standard deviation.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate first: `t` and `y` the same length, `noise_sd` strictly positive, else
`ValueError`. Build the design matrix with the column of ones first and time second; that
order is what makes `mean[0]` the intercept. Invert the prior covariance to get its
precision, add the data's precision (design transposed times design, over the squared
noise), invert the sum to get the posterior covariance, then combine the prior's
precision-weighted mean with the data term. Never divide by the number of readings: an
empty `t` must fall straight through to the prior.
</details>

In [ ]:
def fit_posterior(t: np.ndarray, y: np.ndarray, prior_mean: np.ndarray,
                  prior_cov: np.ndarray, noise_sd: float) -> Posterior:
    """Posterior over (intercept, slope) of the log-degradation line, given readings so far.

    Build the design matrix `[1, t]`, then apply the conjugate Gaussian update above. `t` and
    `y` must be the same length; `noise_sd` must be strictly positive. Raise `ValueError`
    otherwise, because a length mismatch here is a silent off-by-one in the caller and a
    `noise_sd` of zero is a division by zero dressed up as a confident answer.

    An empty `t` is legal and returns the prior unchanged — that is the state of knowledge
    before the unit has been read at all.

    Returns: a `Posterior` whose `mean` has shape (2,) and whose `cov` has shape (2, 2).

    Example:
        >>> vague = np.diag([1e6, 1e6])
        >>> t = np.arange(0.0, 50.0)
        >>> post = fit_posterior(t, -2.0 + 0.01 * t, np.zeros(2), vague, 0.2)
        >>> np.round(post.mean, 6)
        array([-2.  ,  0.01])
        >>> fit_posterior(np.array([]), np.array([]), np.zeros(2), vague, 0.2).mean
        array([0., 0.])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fit_posterior() -> None:
    vague = np.diag([1e6, 1e6])
    t = np.arange(0.0, 60.0)
    post = fit_posterior(t, -2.0 + 0.01 * t, np.zeros(2), vague, 0.2)
    assert np.allclose(post.mean, [-2.0, 0.01], atol=1e-6), (
        f"with a vague prior and noiseless data the posterior mean is the least-squares fit "
        f"[-2.0, 0.01]; got {np.round(post.mean, 5)}. A design matrix of [t, 1] rather than "
        "[1, t] swaps them"
    )
    assert post.cov.shape == (2, 2) and np.allclose(post.cov, post.cov.T), (
        f"cov must be a symmetric 2x2, got shape {post.cov.shape}"
    )
    empty = fit_posterior(np.array([]), np.array([]), np.array([-2.5, 0.008]), vague, 0.2)
    assert np.allclose(empty.mean, [-2.5, 0.008]) and np.allclose(empty.cov, vague), (
        f"with no readings the posterior IS the prior; got mean {np.round(empty.mean, 5)}. "
        "If this raised, your code divides by len(t) somewhere it should not"
    )
    strong = np.diag([1e-8, 1e-12])
    pinned = fit_posterior(np.array([0.0]), np.array([5.0]), np.array([-2.5, 0.008]),
                           strong, 0.2)
    assert abs(pinned.mean[0] + 2.5) < 1e-3, (
        f"one reading against a very tight prior must barely move it; the intercept went to "
        f"{pinned.mean[0]:.3f}. Ignoring prior_cov and returning the plain least-squares fit "
        "passes the first check and fails this one"
    )
    unit_prior = fit_posterior(np.array([0.0]), np.array([5.0]), np.zeros(2),
                               np.diag([1.0, 1.0]), 0.5)
    assert (abs(float(unit_prior.mean[0]) - 4.0) < 1e-9
            and abs(float(unit_prior.cov[0, 0]) - 0.2) < 1e-9), (
        f"one reading of 5.0 at t=0 against a unit prior with noise_sd 0.5 has an exact "
        f"answer: precision 1 + 1/0.25 = 5, so cov[0, 0] is 0.2 and the intercept is 4.0. "
        f"Got {float(unit_prior.mean[0]):.6f} and {float(unit_prior.cov[0, 0]):.6f}; "
        "3.333333 and 0.333333 mean you divided by noise_sd where the formula needs "
        "noise_sd ** 2"
    )
    prior_mean, prior_cov, noise_sd = prior()
    hours = np.arange(0, 400, SAMPLE_EVERY, dtype=float)
    y = log_signal(HEALTH[FLEET_FAILURES[0].unit, hours.astype(int)])
    few = fit_posterior(hours[:6], y[:6], prior_mean, prior_cov, noise_sd)
    many = fit_posterior(hours, y, prior_mean, prior_cov, noise_sd)
    assert np.trace(many.cov) < np.trace(few.cov), (
        f"the posterior must tighten as readings arrive: trace went {np.trace(few.cov):.3e} "
        f"-> {np.trace(many.cov):.3e}. If it grew, X.T @ X is being subtracted rather than "
        "added to the prior precision"
    )
    for bad_t, bad_y, bad_sd in ((np.arange(3.0), np.arange(4.0), 0.2),
                                 (np.arange(3.0), np.arange(3.0), 0.0),
                                 (np.arange(3.0), np.arange(3.0), -1.0)):
        try:
            fit_posterior(bad_t, bad_y, np.zeros(2), vague, bad_sd)
        except ValueError:
            continue
        raise AssertionError(f"t of {bad_t.size}, y of {bad_y.size}, noise_sd {bad_sd} must "
                             "raise ValueError rather than return a confident answer")
    print(f"exercise 2 looks right — on unit {FLEET_FAILURES[0].unit} the slope posterior "
          f"tightened from ±{np.sqrt(few.cov[1, 1]):.5f} to ±{np.sqrt(many.cov[1, 1]):.5f}")

In [ ]:
_try("exercise 2", _check_fit_posterior)

## 5. Exercise 3 — `rul_samples()`, where the error bars come from

You have a Gaussian posterior over `(a, b)`. The unit fails when `a + b*t` reaches
`Y_FAIL`, so the crossing time is `(Y_FAIL - a) / b` and the remaining life at `t_now` is
that minus `t_now`. A ratio of two correlated Gaussians has no tidy closed form, so draw
from the posterior and push each draw through the arithmetic. The spread of what comes out
*is* the predictive distribution.

Two draws need a decision rather than a formula, and both are graded:

- **A draw with `b <= 0`** says the unit is improving and will never reach the failure
  level. There is no crossing time. Report `RUL_CAP` — "longer than this notebook will
  commit to" — rather than a negative number or an infinity that poisons every quantile.
- **A draw with a crossing time already behind you** says the unit should have failed by
  now. Clip it at zero. A predictive distribution with mass below zero is telling you the
  model and the machine disagree, and zero is the honest way to say so.

<details><summary>💡 Hint 1 — what to think about</summary>

Three kinds of draw need a decision rather than the formula: a slope of zero or below (the
line never reaches the failure level, so what does the division hand you instead?), a
crossing time already behind `t_now`, and a posterior so narrow that every draw is the
plug-in answer. For that last one, is your answer counted from hour 0 or from `t_now`? And
would two calls with the same seed return the same draws?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Make a fresh generator from `seed` inside the function and draw intercept-slope pairs from
the full covariance, not from two independent normals. Turn each draw into a crossing hour
using `y_fail` from the argument, then subtract `t_now`. Replace every non-degrading draw
with `RUL_CAP`, which is already an amount of remaining life and so is not shifted by
`t_now`, and finally clip everything into the range from zero to the cap.
</details>

In [ ]:
def rul_samples(mean: np.ndarray, cov: np.ndarray, t_now: float, y_fail: float,
                n_samples: int = N_SAMPLES, seed: int = SEED) -> np.ndarray:
    """Draw a predictive distribution of remaining useful life from a parameter posterior.

    Draw `n_samples` pairs `(a, b)` from `N(mean, cov)` using
    `np.random.default_rng(seed)` — same seed, same draws, so two runs of this notebook agree.
    For each draw the crossing time is `(y_fail - a) / b` and the remaining life is
    `crossing - t_now`. Then: any draw with `b <= 0` becomes `RUL_CAP`, and everything is
    clipped into `[0.0, RUL_CAP]`.

    Returns: an `np.ndarray` of shape (n_samples,), float64.

    Example:
        >>> tiny = np.diag([1e-12, 1e-18])          # a posterior with no uncertainty left
        >>> s = rul_samples(np.array([0.0, 0.01]), tiny, 50.0, 2.0, 200, seed=1)
        >>> float(np.round(s.mean(), 3))            # crossing at 200, so 150 hours left
        150.0
        >>> float(rul_samples(np.array([0.0, -0.01]), tiny, 0.0, 2.0, 10, seed=1).min())
        1500.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_rul_samples() -> None:
    tiny = np.diag([1e-12, 1e-18])
    plug = rul_samples(np.array([0.0, 0.01]), tiny, 50.0, 2.0, 400, seed=1)
    assert plug.shape == (400,) and plug.dtype == np.float64, (
        f"expected 400 float64 draws, got shape {plug.shape} dtype {plug.dtype}"
    )
    assert abs(float(plug.mean()) - 150.0) < 1e-3, (
        f"with a posterior of essentially zero width the draws must all be the plug-in "
        f"answer (2.0 - 0.0)/0.01 - 50 = 150; got {plug.mean():.3f}. A value of 200 means "
        "you forgot to subtract t_now; 50 means you subtracted it from the wrong side"
    )
    again = rul_samples(np.array([0.0, 0.01]), tiny, 50.0, 2.0, 400, seed=1)
    assert np.array_equal(plug, again), (
        "the same seed must give the same draws. Seed a fresh np.random.default_rng(seed) "
        "inside the function rather than using np.random's global state"
    )
    other = rul_samples(np.array([0.0, 0.01]), tiny * 1e12, 50.0, 2.0, 400, seed=2)
    assert not np.array_equal(plug[:10], other[:10]), (
        "a different seed and a real covariance must give different draws; yours are "
        "identical, so the covariance is being ignored"
    )
    improving = rul_samples(np.array([0.0, -0.01]), tiny, 0.0, 2.0, 50, seed=1)
    assert np.allclose(improving, RUL_CAP), (
        f"a draw whose slope is negative never reaches the failure level: report RUL_CAP "
        f"({RUL_CAP:.0f}), got {improving.min():.1f} .. {improving.max():.1f}. A negative "
        "crossing time here is the commonest way to get a nonsense lower quantile"
    )
    still_improving = rul_samples(np.array([0.0, -0.01]), tiny, 400.0, 2.0, 50, seed=1)
    assert np.allclose(still_improving, RUL_CAP), (
        f"the same non-degrading posterior read at t_now=400 must STILL report RUL_CAP "
        f"({RUL_CAP:.0f}); got {still_improving.min():.1f}. {RUL_CAP - 400.0:.0f} means "
        "t_now was subtracted from the cap as well as from the crossing time. RUL_CAP is "
        "already an amount of remaining life, so it is capped, not shifted"
    )
    overdue = rul_samples(np.array([0.0, 0.01]), tiny, 900.0, 2.0, 50, seed=1)
    assert np.allclose(overdue, 0.0), (
        f"the crossing time is hour 200 and t_now is 900, so every draw is overdue and must "
        f"clip to 0.0; got {overdue.min():.1f}. Leaving it negative puts mass below zero in "
        "a quantity that cannot be negative"
    )
    spread = rul_samples(np.array([-1.0, 0.01]), np.diag([0.04, 4e-7]), 100.0, Y_FAIL,
                         2000, seed=3)
    assert 0.0 < float(np.quantile(spread, 0.05)) < float(np.quantile(spread, 0.95)), (
        "a real posterior must give a real spread; your 5th and 95th percentiles are not "
        "ordered, which usually means the draws are not being generated from cov at all"
    )
    print(f"exercise 3 looks right — a mid-life posterior gives a 90% interval "
          f"{np.quantile(spread, 0.05):.0f} to {np.quantile(spread, 0.95):.0f} hours")

In [ ]:
_try("exercise 3", _check_rul_samples)

## 6. Exercise 4 — `rul_track()`, the same question asked every day

A prognostic is not run once. It is run on a schedule, and each run gets one more day of
evidence than the last. `rul_track` is that schedule: at each checkpoint hour, fit the
posterior on everything read **up to and including that hour**, draw the predictive
distribution, and record the quantiles you care about.

The word to read twice is *up to*. The obvious implementation fits the whole row and then
asks about hour 200; on this fleet that quietly hands the model the repair, the overhaul
and the replacement unit. It is module 4's leak in a new costume, and the rubric puts a
nonsense tail on the row specifically to catch it.

<details><summary>💡 Hint 1 — what to think about</summary>

The row you are given runs past every checkpoint, and on this fleet the hours after a
checkpoint can belong to a repaired or replaced machine. Which readings is the prognostic
at hour `t_k` actually allowed to see, and is the reading taken AT `t_k` one of them?
Then: what tells `rul_samples` how much of the unit's life has already gone?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over the checkpoints with their index `k`. For each one, read the row from hour 0 up
to and including `t_k` at the sampling interval, fit the posterior on only those readings,
draw with `t_now` set to `t_k` and the seed moved on by `k`, and take all the requested
quantiles of that one set of draws. Stack one row per checkpoint, so the result is
checkpoints by quantiles and not its transpose.
</details>

In [ ]:
def rul_track(y_row: np.ndarray, checkpoints: np.ndarray, prior_mean: np.ndarray,
              prior_cov: np.ndarray, noise_sd: float, y_fail: float = Y_FAIL,
              quantiles: np.ndarray = QUANTILES, sample_every: int = SAMPLE_EVERY,
              n_samples: int = N_SAMPLES, seed: int = SEED) -> np.ndarray:
    """Re-predict remaining useful life at every checkpoint, using only the past.

    For each checkpoint `k` at hour `t_k`:
      * read hours `np.arange(0, t_k + 1, sample_every)` of `y_row` — and nothing after `t_k`;
      * `fit_posterior` on those readings;
      * `rul_samples` from that posterior at `t_now = t_k`, with `seed + k` so each
        checkpoint draws its own reproducible sample;
      * take `np.quantile` of the draws at `quantiles`.

    Returns: an `np.ndarray` of shape `(len(checkpoints), len(quantiles))`, one row per
    checkpoint, each row non-decreasing across the quantiles.

    Example:
        >>> y = -2.0 + 0.01 * np.arange(400.0)
        >>> track = rul_track(y, np.array([200, 300]), np.zeros(2), np.diag([1e6, 1e6]),
        ...                   0.05, y_fail=2.0, quantiles=np.array([0.5]))
        >>> track.shape
        (2, 1)
        >>> bool(track[1, 0] < track[0, 0])         # less life left a hundred hours later
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_rul_track() -> None:
    clean = -2.0 + 0.01 * np.arange(400.0)
    qs = np.array([0.05, 0.5, 0.95])
    track = rul_track(clean, np.array([200, 300]), np.zeros(2), np.diag([1e6, 1e6]), 0.05,
                      y_fail=2.0, quantiles=qs)
    assert track.shape == (2, 3), (
        f"expected one row per checkpoint and one column per quantile, (2, 3); got "
        f"{track.shape}. A transposed result breaks every metric in section 8"
    )
    assert np.all(np.diff(track, axis=1) >= 0), (
        "each row must be non-decreasing across the quantiles — np.quantile guarantees that, "
        "so an out-of-order row means the draws are being re-generated per quantile"
    )
    assert track[1, 1] < track[0, 1], (
        f"a hundred hours later there is less life left: median went {track[0, 1]:.1f} -> "
        f"{track[1, 1]:.1f}. If it rose, t_now is not being passed through to rul_samples"
    )
    dirty = clean.copy()
    dirty[201:] = 50.0                     # the overhauled unit, or a sensor swap
    same = rul_track(dirty, np.array([200]), np.zeros(2), np.diag([1e6, 1e6]), 0.05,
                     y_fail=2.0, quantiles=qs)
    assert np.allclose(same[0], track[0]), (
        f"the prediction at hour 200 changed when hours 201+ changed: {np.round(track[0], 1)} "
        f"-> {np.round(same[0], 1)}. You are fitting the whole row. Everything after the "
        "checkpoint is the future, and on this fleet it is a different machine"
    )
    coarse = rul_track(clean, np.array([200]), np.zeros(2), np.diag([1e6, 1e6]), 0.05,
                       y_fail=2.0, quantiles=qs, sample_every=200)
    assert abs(float(coarse[0, 1]) - 200.0) < 25.0, (
        f"with sample_every=200 the readings are hours 0 AND 200, which pin this ramp to a "
        f"crossing at hour 400 and 200 hours left; got a median of {coarse[0, 1]:.0f}. The "
        "checkpoint hour itself is included: np.arange(0, t_k + 1, sample_every)"
    )
    repeat = rul_track(clean, np.array([200, 300]), np.zeros(2), np.diag([1e6, 1e6]), 0.05,
                       y_fail=2.0, quantiles=qs)
    assert np.array_equal(track, repeat), "the same inputs must give the same track"
    varied = rul_track(clean, np.array([200, 300]), np.zeros(2), np.diag([1.0, 1e-4]), 0.3,
                       y_fail=2.0, quantiles=qs)
    assert not np.allclose(varied[0], varied[1]), (
        "two checkpoints must not draw the identical sample; pass `seed + k` so each "
        "checkpoint has its own stream"
    )
    print(f"exercise 4 looks right — on a clean ramp the 90% interval went "
          f"{track[0, 2] - track[0, 0]:.0f} h wide at hour 200 to "
          f"{track[1, 2] - track[1, 0]:.0f} h at hour 300")

In [ ]:
_try("exercise 4", _check_rul_track)

Now run it on the fleet. `fleet_predictions()` below is given: it calls YOUR `rul_track`
once per failed fleet unit, at every checkpoint from hour 96 to the hour the unit stopped,
and stacks the answers into one table. Every number in the rest of this notebook comes out
of that table.

In [ ]:
_PREDICTIONS: list = []
# fleet_predictions() runs your first four exercises end to end, so every cell that
# reads its table waits until all four have passed their checks.
_FOR_TRACKS = ("exercise 1", "exercise 2", "exercise 3", "exercise 4")


def checkpoints_for(run: Run) -> np.ndarray:
    """The hours at which the prognostic is re-run for one unit. Given to you."""
    return np.arange(FIRST_CHECK, run.end_hour, CHECK_EVERY)


def fleet_predictions() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Every prediction this fleet generates, computed once. Depends on YOUR rul_track.

    Returns `(unit, hour, true_rul, quantile_matrix)`, all aligned row by row. The matrix has
    one column per entry of `QUANTILES`, so `matrix[:, i]` is one complete model: "always act
    on the QUANTILES[i] quantile of the predictive distribution".
    """
    if not _PREDICTIONS:
        prior_mean, prior_cov, noise_sd = prior()
        signal = log_signal(HEALTH)
        units, hours, true, blocks = [], [], [], []
        for run in FLEET_FAILURES:
            checks = checkpoints_for(run)
            blocks.append(rul_track(signal[run.unit], checks, prior_mean, prior_cov, noise_sd,
                                    seed=SEED + 1000 * run.unit))
            units.append(np.full(checks.size, run.unit))
            hours.append(checks.astype(float))
            true.append(run.end_hour - checks.astype(float))
        _PREDICTIONS.append((np.concatenate(units), np.concatenate(hours),
                             np.concatenate(true), np.vstack(blocks)))
    return _PREDICTIONS[0]


def _show_narrowing() -> None:
    _, hours, true, matrix = fleet_predictions()
    lo = matrix[:, 0]
    hi = matrix[:, -1]
    width = hi - lo
    elapsed = hours / (hours + true)
    print(f"{len(true):,} predictions over {len(FLEET_FAILURES)} units that ran to failure")
    print(f"{'life elapsed':<14}{'rows':>6}{'median true RUL':>18}{'90% interval':>15}"
          f"{'as % of RUL':>14}")
    bands = ((0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.01))
    widths = []
    for low, high in bands:
        keep = (elapsed >= low) & (elapsed < high)
        if not keep.any():
            continue
        widths.append(float(np.median(width[keep])))
        print(f"{f'{low:.0%}-{high:.0%}':<14}{int(keep.sum()):>6}"
              f"{np.median(true[keep]):>18.0f}{widths[-1]:>13.0f} h"
              f"{np.median(width[keep] / np.maximum(true[keep], 1.0)):>13.0%}")
    print(f"\nthe interval narrows from {widths[0]:.0f} h to {widths[-1]:.0f} hours wide, a "
          f"factor of {widths[0] / widths[-1]:.1f}, as the evidence accumulates.")
    print("It does NOT narrow as a percentage of the remaining life, and the last column says "
          "so:\nnear the end there is less life left to be uncertain about.")


_try("the interval narrows", _show_narrowing, needs=_FOR_TRACKS)

In [ ]:
def _plot_one_track() -> None:
    _, hours, true, matrix = fleet_predictions()
    run = max(FLEET_FAILURES, key=lambda r: r.end_hour)
    keep = fleet_predictions()[0] == run.unit
    t = hours[keep]
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    axes[0].plot(np.arange(run.end_hour), HEALTH[run.unit, :run.end_hour], lw=0.8)
    axes[0].axhline(FAILURE_LEVEL, color="black", ls="--", lw=1.0, label="failure level")
    axes[0].set_xlabel("hour"), axes[0].set_ylabel("health index")
    axes[0].set_title(f"unit {run.unit}: the evidence")
    axes[0].legend()
    mid = QUANTILES.tolist().index(0.50)
    axes[1].fill_between(t, matrix[keep, 0], matrix[keep, -1], alpha=0.25,
                         label=f"{QUANTILES[0]:.0%}-{QUANTILES[-1]:.0%} predictive interval")
    axes[1].plot(t, matrix[keep, mid], lw=1.5, label="predicted RUL, median")
    axes[1].plot(t, true[keep], color="black", lw=1.5, label="true RUL")
    axes[1].set_xlabel("hour"), axes[1].set_ylabel("remaining useful life, hours")
    axes[1].set_title("the answer, re-asked every 24 hours")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)
    print(f"plotted: unit {run.unit}, {int(keep.sum())} checkpoints, interval closing on the "
          f"truth")


_try("the track, plotted", _plot_one_track, needs=_FOR_TRACKS)

## 7. Exercise 5 — `rul_rmse()`, the number everybody reports

Every RUL paper reports it and every benchmark ranks on it. It is also the one metric in
this notebook that cannot tell the difference between being forty hours early and forty
hours late. Implement it first, so that when it picks the wrong model in section 11 it is
your own implementation doing the picking.

The evaluation set needs a boundary too. A prognosis at 900 hours to go is arithmetic, not
prognostics; nobody schedules anything on it and its error swamps every average. This
notebook scores only the rows with `true_rul <= EVAL_HORIZON`, and every metric in
sections 7 to 11 is computed on exactly those rows — the comparison is only honest if the
metrics are looking at the same table.

<details><summary>💡 Hint 1 — what to think about</summary>

Three averages look alike here: the mean of the signed errors, the mean of their sizes,
and the root of the mean of their squares. One prediction early and one equally late tells
them apart: which of the three says that pair is perfect? And what should an RMSE over
zero rows be, given that the answer will be fed to a minimisation?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Refuse mismatched lengths and an empty table with `ValueError`. Then square each error,
average over n (not n - 1), and take the square root last. Return a plain Python float.
Resist the urge to make it asymmetric; this metric is meant to be blind to direction, and
section 11 depends on it being so.
</details>

In [ ]:
def rul_rmse(predicted: np.ndarray, true_rul: np.ndarray) -> float:
    """Root mean squared error between predicted and true remaining useful life, in hours.

    Raise `ValueError` if the two arrays differ in length or if there is nothing to score:
    an RMSE over zero rows is a nan that will win any minimisation you feed it to.

    Returns: a Python `float`.

    Example:
        >>> float(np.round(rul_rmse(np.array([90.0, 110.0]), np.array([100.0, 100.0])), 6))
        10.0
        >>> float(rul_rmse(np.array([100.0]), np.array([100.0])))
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_rul_rmse() -> None:
    got = rul_rmse(np.array([90.0, 110.0]), np.array([100.0, 100.0]))
    assert abs(got - 10.0) < 1e-9, (
        f"one prediction 10 hours early and one 10 hours late is an RMSE of 10.0, got {got}. "
        "A value of 0.0 means you averaged the signed errors before squaring them"
    )
    assert abs(rul_rmse(np.array([60.0]), np.array([100.0])) - 40.0) < 1e-9, (
        "a single row 40 hours early has an RMSE of 40"
    )
    early = rul_rmse(np.array([60.0]), np.array([100.0]))
    late = rul_rmse(np.array([140.0]), np.array([100.0]))
    assert abs(early - late) < 1e-9, (
        f"40 hours early scores {early} and 40 hours late scores {late}. RMSE is SYMMETRIC — "
        "if yours is not, you have already fixed the bug this lesson is about, in the wrong "
        "place. Keep it symmetric; section 8 is where asymmetry belongs"
    )
    for bad in ((np.array([1.0, 2.0]), np.array([1.0])), (np.array([]), np.array([]))):
        try:
            rul_rmse(*bad)
        except ValueError:
            continue
        raise AssertionError("mismatched lengths and an empty table must both raise "
                             "ValueError; an RMSE of nan silently wins every argmin")
    print("exercise 5 looks right — and deliberately blind to the sign of the error")

In [ ]:
_try("exercise 5", _check_rul_rmse)

## 8. Exercise 6 — `phm_score()`, a metric that knows which way is dangerous

The C-MAPSS turbofan challenge scored entrants with an asymmetric exponential. Writing
`d = predicted - true` — positive means the prognosis promised more life than the unit had,
which is *late* — the per-unit score is

> `exp(-d / EARLY_TAU) - 1` when `d < 0`, and `exp(d / LATE_TAU) - 1` when `d >= 0`

with `LATE_TAU = 10` and `EARLY_TAU = 13`. The smaller time constant on the late side is
the whole design: the same number of hours costs more when you find out too late. A perfect
prediction scores 0 and every error scores positive, so lower is better.

Two things about it are worth having opinions on. It is exponential, so one badly late
prediction can outweigh a hundred good ones — that is deliberate, and it is why section 11
also reports a plain count. And the challenge summed it over units; this notebook takes the
mean instead, so that tables computed over different numbers of rows stay comparable.

<details><summary>💡 Hint 1 — what to think about</summary>

Fix the sign convention before anything else. `d` is predicted minus true, so a positive
`d` promised more life than the unit had: that is late. Which time constant belongs on the
late branch, the larger or the smaller? And what should an exactly right prediction score,
and can any score ever be negative?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compute `d` for every row, then choose between the two exponential branches row by row
with a vectorised selection, not a Python `if` on the whole array. Subtract one after
exponentiating, on both branches, so a perfect prediction scores zero. Return the per-row
array, not its sum or mean (`mean_phm_score` does the averaging), and raise `ValueError`
on a length mismatch rather than letting numpy broadcast.
</details>

In [ ]:
def phm_score(predicted: np.ndarray, true_rul: np.ndarray) -> np.ndarray:
    """The C-MAPSS asymmetric prognostic score, one value per row. Lower is better.

    `d = predicted - true_rul`. Return `exp(-d / EARLY_TAU) - 1` where `d < 0` and
    `exp(d / LATE_TAU) - 1` where `d >= 0`. Raise `ValueError` on a length mismatch.

    Returns: an `np.ndarray` of shape `(n,)`, float64, every element >= 0.

    Example:
        >>> float(phm_score(np.array([100.0]), np.array([100.0]))[0])
        0.0
        >>> float(np.round(phm_score(np.array([110.0]), np.array([100.0]))[0], 6))
        1.718282
        >>> float(np.round(phm_score(np.array([90.0]), np.array([100.0]))[0], 6))
        1.158193
    """
    # YOUR CODE HERE
    raise NotImplementedError


def mean_phm_score(predicted: np.ndarray, true_rul: np.ndarray) -> float:
    """The mean of YOUR phm_score over the rows. Given to you; lower is better."""
    return float(np.mean(phm_score(predicted, true_rul)))


def _check_phm_score() -> None:
    exact = phm_score(np.array([100.0]), np.array([100.0]))
    assert exact.shape == (1,) and abs(float(exact[0])) < 1e-12, (
        f"an exactly right prediction scores 0, got {exact}. Return one score per row, not "
        "the total"
    )
    late = float(phm_score(np.array([110.0]), np.array([100.0]))[0])
    early = float(phm_score(np.array([90.0]), np.array([100.0]))[0])
    assert abs(late - (np.exp(1.0) - 1.0)) < 1e-9, (
        f"ten hours late is exp(10/{LATE_TAU:.0f}) - 1 = {np.exp(1.0) - 1.0:.6f}, "
        f"got {late:.6f}"
    )
    assert abs(early - (np.exp(10.0 / EARLY_TAU) - 1.0)) < 1e-9, (
        f"ten hours early is exp(10/{EARLY_TAU:.0f}) - 1 = "
        f"{np.exp(10.0 / EARLY_TAU) - 1.0:.6f}, got {early:.6f}"
    )
    assert late > early, (
        f"ten hours late ({late:.3f}) must cost more than ten hours early ({early:.3f}). If "
        "they match you used one time constant for both branches; if it is the other way "
        "round you have swapped LATE_TAU and EARLY_TAU, which inverts the lesson"
    )
    signs = phm_score(np.array([40.0, 100.0, 260.0]), np.array([100.0, 100.0, 100.0]))
    assert (signs >= 0).all(), (
        f"every score is >= 0 because exp(x) >= 1 on both branches; got {np.round(signs, 3)}. "
        "A negative value means the -1 was applied outside the exponential"
    )
    try:
        phm_score(np.array([1.0, 2.0]), np.array([1.0]))
    except ValueError:
        pass
    else:
        raise AssertionError("a length mismatch must raise ValueError, not broadcast")
    big_late = float(phm_score(np.array([160.0]), np.array([100.0]))[0])
    big_early = float(phm_score(np.array([40.0]), np.array([100.0]))[0])
    print(f"exercise 6 looks right — ten hours late costs {late / early:.2f}x ten hours "
          f"early, and sixty hours late costs {big_late / big_early:.1f}x sixty hours early")

In [ ]:
_try("exercise 6", _check_phm_score)

## 9. Exercise 7 — `cone_breakdown()`, and the alpha-lambda accuracy cone

The standard prognostic accuracy metric does not use a fixed tolerance in hours. It uses a
band that is a *fraction* of the true remaining life, so the requirement tightens as the
unit approaches failure: being 40 hours out with 400 to go is fine, and being 40 hours out
with 50 to go is not. Plotted against time it is a cone closing on zero.

A prediction is inside the cone when `(1 - alpha) * true <= predicted <= (1 + alpha) * true`,
inclusive at both ends. Return all three rates, not just the middle one, because the two
outer ones are the point of this lesson: the cone is symmetric, so a model that is always
a little early and a model that is always a little late score exactly the same.

<details><summary>💡 Hint 1 — what to think about</summary>

The cone's half-width is a fraction of each row's TRUE remaining life, so the same miss in
hours can be inside the cone on one row and outside it on another. Which side is early:
below the cone or above it? And does a prediction sitting exactly on a bound count as
inside?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate `alpha` from the argument (strictly between 0 and 1), equal lengths and at least
one row. Give each row its own lower and upper bound from its own true RUL. Count a row as
early only when it is strictly below the lower bound and late only when it is strictly
above the upper one, so the bounds themselves fall inside. Divide the counts by the number
of rows and return three floats in the order early, inside, late.
</details>

In [ ]:
def cone_breakdown(predicted: np.ndarray, true_rul: np.ndarray,
                   alpha: float = ALPHA) -> tuple[float, float, float]:
    """Split predictions into early, inside and late relative to the alpha accuracy cone.

    The cone at a row with true remaining life `r` runs from `(1 - alpha) * r` to
    `(1 + alpha) * r`, inclusive at both bounds. Below it the prognosis is *early* (it
    understated the life left); above it, *late*. Raise `ValueError` unless
    `0 < alpha < 1`, and on a length mismatch or an empty table.

    Returns: `(early_rate, inside_rate, late_rate)`, three floats summing to 1.0.

    Example:
        >>> cone_breakdown(np.array([80.0]), np.array([100.0]), 0.2)   # exactly on the bound
        (0.0, 1.0, 0.0)
        >>> cone_breakdown(np.array([79.0]), np.array([100.0]), 0.2)
        (1.0, 0.0, 0.0)
        >>> cone_breakdown(np.array([79.0, 121.0]), np.array([100.0, 100.0]), 0.2)
        (0.5, 0.0, 0.5)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def cone_accuracy(predicted: np.ndarray, true_rul: np.ndarray, alpha: float = ALPHA) -> float:
    """The alpha-lambda accuracy itself: the inside rate. Given to you; higher is better."""
    return cone_breakdown(predicted, true_rul, alpha)[1]


def _check_cone_breakdown() -> None:
    on_bound = cone_breakdown(np.array([80.0, 120.0]), np.array([100.0, 100.0]), 0.2)
    assert on_bound == (0.0, 1.0, 0.0), (
        f"predictions exactly on the bounds are INSIDE the cone; got {on_bound}. A strict "
        "`<` on both sides drops them and quietly moves your accuracy down"
    )
    outside = cone_breakdown(np.array([79.0, 121.0]), np.array([100.0, 100.0]), 0.2)
    assert outside == (0.5, 0.0, 0.5), (
        f"one just below the cone is early and one just above is late; got {outside}. If "
        "they came back the other way round you have the two ends swapped, which reverses "
        "every conclusion in section 11"
    )
    scaled = cone_breakdown(np.array([90.0, 90.0, 90.0, 170.0, 1100.0]),
                            np.array([100.0, 50.0, 400.0, 200.0, 1000.0]), 0.2)
    assert np.allclose(scaled, (0.2, 0.6, 0.2)), (
        f"the cone is a fraction of the true RUL, not a fixed number of hours: 90 is inside "
        f"for a true RUL of 100, late for 50 and early for 400, while 170 is inside for 200 "
        f"and 1100 is inside for 1000. Expected (0.2, 0.6, 0.2), got "
        f"{tuple(round(v, 3) for v in scaled)} — no fixed tolerance in hours can do that"
    )
    rates = cone_breakdown(np.array([10.0, 100.0, 300.0]), np.array([100.0] * 3), 0.2)
    assert abs(sum(rates) - 1.0) < 1e-12, f"the three rates must sum to 1.0, got {sum(rates)}"
    for bad_alpha in (0.0, 1.0, -0.2, 1.5):
        try:
            cone_breakdown(np.array([1.0]), np.array([1.0]), bad_alpha)
        except ValueError:
            continue
        raise AssertionError(f"alpha={bad_alpha} is not a fraction strictly between 0 and 1 "
                             "and must raise ValueError")
    for bad in ((np.array([1.0, 2.0]), np.array([1.0])), (np.array([]), np.array([]))):
        try:
            cone_breakdown(*bad, 0.2)
        except ValueError:
            continue
        raise AssertionError("a length mismatch and an empty table must both raise ValueError")
    print(f"exercise 7 looks right — at alpha {ALPHA:.0%} the cone is "
          f"±{ALPHA * 100:.0f}% of whatever life is actually left")

In [ ]:
_try("exercise 7", _check_cone_breakdown)

In [ ]:
def _plot_the_cone() -> None:
    units, hours, true, matrix = fleet_predictions()
    keep = true <= EVAL_HORIZON
    order = np.argsort(true[keep])
    ref = true[keep][order]
    mid = QUANTILES.tolist().index(0.50)
    low = QUANTILES.tolist().index(0.10)
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    ax.fill_between(ref, (1 - ALPHA) * ref, (1 + ALPHA) * ref, alpha=0.25, color="grey",
                    label=f"alpha cone, ±{ALPHA:.0%} of true RUL")
    ax.plot(ref, ref, color="black", lw=1.2, label="perfect")
    ax.scatter(true[keep], matrix[keep, mid], s=5, alpha=0.4,
               label=f"act on the {QUANTILES[mid]:.0%} quantile")
    ax.scatter(true[keep], matrix[keep, low], s=5, alpha=0.4,
               label=f"act on the {QUANTILES[low]:.0%} quantile")
    ax.set_xlabel("true remaining useful life, hours")
    ax.set_ylabel("predicted remaining useful life, hours")
    ax.set_title("the accuracy cone closes as the unit approaches failure")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)
    for i in (mid, low):
        e, c, l = cone_breakdown(matrix[keep, i], true[keep], ALPHA)
        name = f"{QUANTILES[i]:.0%} quantile"
        print(f"{name:<16} inside the cone {c:.1%} · early {e:.1%} · late {l:.1%}")
    print("Two models. Read the middle column and they are the same model; read the last one "
          "and\nthey are not.")


_try("the cone, plotted", _plot_the_cone, needs=_FOR_TRACKS + ("exercise 7",))

## 10. Exercise 8 — `warning_lead()`, the metric the workshop cares about

None of the three metrics so far mentions what the number is *for*. A prognosis exists to
raise a job with enough notice to do it: the fitters need `LEAD_HOURS` and you raise the
work order the first time the model says remaining life has fallen to `ACTION_RUL`.

So the operational question is not "how big was the error" but "how many hours of warning
did this model actually buy". Implement that, and note the word **first**: a track that
dips below the action level, rises again on the next reading and dips again is still a
warning at the first dip, because that is when the job was raised.

<details><summary>💡 Hint 1 — what to think about</summary>

The work order is raised once: the first time the track says remaining life has fallen to
the action level. Not at the lowest prediction, and not at the last dip. What if the track
never gets there: is "no warning at all" the same thing as "a warning with no notice"? And
does a prediction exactly at the action level trigger the job?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Refuse empty or mismatched inputs with `ValueError`. Find the first checkpoint whose
predicted RUL is at or below `action_rul`, taken from the argument, and return the failure
hour minus that checkpoint's hour as a float. If no checkpoint qualifies, return the
sentinel the docstring names for no warning, never zero: a zero reads as a warning with
no notice, which is a different failure.
</details>

In [ ]:
def warning_lead(checkpoints: np.ndarray, predicted_rul: np.ndarray, end_hour: int,
                 action_rul: float = ACTION_RUL) -> float:
    """Hours of warning one unit's prognostic track bought, before it actually failed.

    Find the FIRST checkpoint whose predicted remaining life is `<= action_rul` — that is
    when the work order is raised — and return `end_hour` minus that checkpoint's hour. If
    no checkpoint ever reaches the action level, return `-1.0`: no warning was given at all.
    Raise `ValueError` if the two arrays differ in length or are empty.

    Returns: a Python `float` — hours of warning, or -1.0.

    Example:
        >>> warning_lead(np.array([100, 124, 148]), np.array([200.0, 80.0, 40.0]), 200, 72.0)
        52.0
        >>> warning_lead(np.array([100, 124]), np.array([200.0, 180.0]), 200, 72.0)
        -1.0
        >>> warning_lead(np.array([100, 124]), np.array([72.0, 10.0]), 200, 72.0)
        100.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_warning_lead() -> None:
    got = warning_lead(np.array([100, 124, 148]), np.array([200.0, 80.0, 40.0]), 200, 72.0)
    assert got == 52.0, (
        f"the first checkpoint at or below 72 is hour 148, so the warning came 52 hours "
        f"before the failure at hour 200; got {got}"
    )
    none = warning_lead(np.array([100, 124]), np.array([200.0, 180.0]), 200, 72.0)
    assert none == -1.0, (
        f"this track never reaches the action level, so there was no warning: return -1.0, "
        f"got {none}. Returning 0.0 makes 'no warning at all' look like 'a warning with no "
        "notice', and they are different failures"
    )
    on_bound = warning_lead(np.array([100, 124]), np.array([72.0, 10.0]), 200, 72.0)
    assert on_bound == 100.0, (
        f"exactly at the action level counts — the test is `<=`, got {on_bound}"
    )
    noisy = warning_lead(np.array([100, 124, 148]), np.array([60.0, 200.0, 30.0]), 200, 72.0)
    assert noisy == 100.0, (
        f"the job is raised at the FIRST dip below the action level, hour 100, whatever the "
        f"track does afterwards; got {noisy}. Searching for the last crossing, or for the "
        "minimum, reports a warning the plant never acted on"
    )
    for bad in ((np.array([1, 2]), np.array([1.0])), (np.array([]), np.array([]))):
        try:
            warning_lead(bad[0], bad[1], 100, 72.0)
        except ValueError:
            continue
        raise AssertionError("mismatched lengths and an empty track must raise ValueError")
    print(f"exercise 8 looks right — the contract needs {LEAD_HOURS:.0f} h and the job is "
          f"raised at a predicted {ACTION_RUL:.0f} h")

In [ ]:
_try("exercise 8", _check_warning_lead)

## 11. Exercise 9 — `best_quantile()`, and the demonstration

You have one predictive distribution per prediction and nineteen ways to turn it into a
number. Each quantile is a different model with the same beliefs: "always act on the 5%
quantile" is a cautious engineer, "always act on the 95%" is an optimistic one, and the
median is what a benchmark submission usually reports.

Sweep all nineteen under a metric and return the winner. Then let four metrics each pick
their favourite, and read what they chose.

<details><summary>💡 Hint 1 — what to think about</summary>

One column of the matrix is one complete model, so the metric has to see a whole column
against the whole truth vector, never a single row. Which way is better depends on `mode`:
does your code ever maximise? And when two columns tie, which one should win, and why is
that the safe choice?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate first: `mode` must be exactly one of the two strings the docstring names, and the
matrix needs one column per quantile and one row per truth. Score each column with the
metric, then take the position of the best score, the first minimum or the first maximum,
which numpy's own arg functions already give you. Return the quantile at that position
(not the position itself) and its score, both as Python floats.
</details>

In [ ]:
def best_quantile(quantiles: np.ndarray, matrix: np.ndarray, true_rul: np.ndarray,
                  metric: Callable[[np.ndarray, np.ndarray], float],
                  mode: str = "min") -> tuple[float, float]:
    """Score every column of a prediction matrix under one metric and return the winner.

    `matrix` has one row per prediction and one column per entry of `quantiles`, so column
    `i` is the model "always act on the `quantiles[i]` quantile". Evaluate
    `metric(matrix[:, i], true_rul)` for each column and return the best, where `mode` is
    `"min"` (lower is better, e.g. RMSE) or `"max"` (higher is better, e.g. cone accuracy).
    On a tie, prefer the EARLIEST column, which on an ascending quantile grid is the more
    cautious model — a coin toss is not a reason to act later.

    Raise `ValueError` if the shapes disagree or `mode` is anything else.

    Returns: `(quantile, value)`, two Python floats.

    Example:
        >>> q = np.array([0.1, 0.5, 0.9])
        >>> m = np.array([[80.0, 100.0, 130.0], [80.0, 100.0, 130.0]])
        >>> best_quantile(q, m, np.array([100.0, 100.0]), rul_rmse, "min")
        (0.5, 0.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_best_quantile() -> None:
    q = np.array([0.1, 0.5, 0.9])
    m = np.array([[70.0, 100.0, 130.0], [70.0, 100.0, 130.0]])
    truth = np.array([100.0, 100.0])
    picked, value = best_quantile(q, m, truth, rul_rmse, "min")
    assert (picked, round(value, 9)) == (0.5, 0.0), (
        f"the middle column is exactly right, so 'min' on RMSE must return (0.5, 0.0); got "
        f"({picked}, {value}). Returning the column INDEX rather than the quantile is the "
        "usual slip"
    )
    hi, acc = best_quantile(q, m, truth, cone_accuracy, "max")
    assert (hi, acc) == (0.5, 1.0), (
        f"only the middle column is inside the ±20% cone, so 'max' on cone accuracy must "
        f"return (0.5, 1.0); got ({hi}, {acc}). If you got 0.1 you are minimising whatever "
        "mode says — a metric where higher is better needs argmax"
    )
    tied = np.array([[90.0, 110.0, 130.0]])
    first, _ = best_quantile(q, tied, np.array([100.0]), rul_rmse, "min")
    assert first == 0.1, (
        f"columns 0 and 1 are both 10 hours out, so the tie goes to the earlier, more "
        f"cautious quantile 0.1; got {first}. np.argmin already does this — np.argsort "
        "followed by a max does not"
    )
    for bad_mode in ("MIN", "lowest", "", None):
        try:
            best_quantile(q, m, truth, rul_rmse, bad_mode)
        except ValueError:
            continue
        raise AssertionError(f"mode={bad_mode!r} must raise ValueError rather than silently "
                             "minimising a metric that should be maximised")
    try:
        best_quantile(np.array([0.1, 0.5]), m, truth, rul_rmse, "min")
    except ValueError:
        pass
    else:
        raise AssertionError("a matrix with more columns than quantiles must raise ValueError")
    try:
        best_quantile(q, m, np.array([100.0]), rul_rmse, "min")
    except ValueError:
        pass
    else:
        raise AssertionError("a matrix with more rows than truths must raise ValueError")
    print("exercise 9 looks right — nineteen models, one sweep, ties broken towards caution")

In [ ]:
_try("exercise 9", _check_best_quantile)

The payoff. Four metrics, one table of predictions, and the question each of them is really
asking. Two of the four are symmetric in the sign of the error. Watch what they choose.

In [ ]:
_SWEEP: list = []
# metric_sweep() scores that table with your four metrics as well.
_FOR_SWEEP = _FOR_TRACKS + ("exercise 5", "exercise 6", "exercise 7", "exercise 8")


def metric_sweep() -> dict:
    """Every metric at every quantile, on the evaluation window. Uses YOUR implementations."""
    if not _SWEEP:
        units, hours, true, matrix = fleet_predictions()
        keep = true <= EVAL_HORIZON
        pred, truth = matrix[keep], true[keep]
        n_q = QUANTILES.size
        leads = np.empty((len(FLEET_FAILURES), n_q))
        for j, run in enumerate(FLEET_FAILURES):
            rows = units == run.unit
            for i in range(n_q):
                leads[j, i] = warning_lead(hours[rows], matrix[rows, i], run.end_hour,
                                           ACTION_RUL)
        _SWEEP.append({
            "pred": pred, "truth": truth, "rows": int(keep.sum()),
            "rmse": np.array([rul_rmse(pred[:, i], truth) for i in range(n_q)]),
            "score": np.array([mean_phm_score(pred[:, i], truth) for i in range(n_q)]),
            "inside": np.array([cone_breakdown(pred[:, i], truth, ALPHA)[1]
                                for i in range(n_q)]),
            "late": np.array([cone_breakdown(pred[:, i], truth, ALPHA)[2]
                              for i in range(n_q)]),
            "short": np.array([float(np.sum(leads[:, i] < LEAD_HOURS)) for i in range(n_q)]),
            "leads": leads,
        })
    return _SWEEP[0]


def _the_wrong_metric_wins() -> None:
    s = metric_sweep()
    pred, truth = s["pred"], s["truth"]
    n_units = len(FLEET_FAILURES)
    choices = (
        ("RMSE on RUL", *best_quantile(QUANTILES, pred, truth, rul_rmse, "min")),
        ("C-MAPSS score", *best_quantile(QUANTILES, pred, truth, mean_phm_score, "min")),
        ("alpha-lambda accuracy", *best_quantile(QUANTILES, pred, truth, cone_accuracy, "max")),
    )
    print(f"{s['rows']:,} predictions inside {EVAL_HORIZON:.0f} hours of failure, over "
          f"{n_units} units\n")
    print(f"{'metric doing the picking':<24}{'picks q':>9}{'RMSE h':>9}{'score':>12}"
          f"{'in cone':>9}{'late':>8}{'short warnings':>16}")
    for name, q, _ in choices:
        i = QUANTILES.tolist().index(round(q, 2))
        warned = "{} of {}".format(int(s["short"][i]), n_units)
        print(f"{name:<24}{q:>9.2f}{s['rmse'][i]:>9.2f}{s['score'][i]:>12.4g}"
              f"{s['inside'][i]:>9.1%}{s['late'][i]:>8.1%}{warned:>16}")
    sym = [c for c in choices if c[0] != "C-MAPSS score"]
    asym_q = choices[1][1]
    print(f"\nThe two symmetric metrics picked {sym[0][1]:.2f} and {sym[1][1]:.2f}. The one "
          f"that prices lateness differently picked {asym_q:.2f}.")

    # The pairing that makes the point without any choosing on my part: the LATEST model in
    # the sweep whose RMSE is still no worse than the most cautious model's.
    cautious = 0
    twin = max(i for i in range(QUANTILES.size) if s["rmse"][i] <= s["rmse"][cautious])
    safe_rmse = s["rmse"][cautious]
    print(f"\nOf the {QUANTILES.size} models you swept, acting on the "
          f"{QUANTILES[twin]:.0%} quantile scores an RMSE of {s['rmse'][twin]:.2f} h, which "
          f"is LOWER than the\n{QUANTILES[cautious]:.0%} quantile's {safe_rmse:.2f} h. "
          f"Ranked on RMSE alone, the late model wins and you ship it.")
    late_ratio = s["late"][twin] / max(s["late"][cautious], 1e-9)
    print(f"  late beyond the cone   {s['late'][twin]:>8.1%}  against "
          f"{s['late'][cautious]:>8.1%}   ({late_ratio:,.0f}x)")
    print(f"  asymmetric score       {s['score'][twin]:>8.4g}  against "
          f"{s['score'][cautious]:>8.4g}   ({s['score'][twin] / s['score'][cautious]:,.0f}x)")
    print(f"  warning shorter than {LEAD_HOURS:.0f} h"
          f"{int(s['short'][twin]):>5} of {n_units}  against {int(s['short'][cautious]):>3} "
          f"of {n_units}")
    print(f"  median warning         {np.median(s['leads'][:, twin]):>8.0f} h  against "
          f"{np.median(s['leads'][:, cautious]):>8.0f} h")
    print("\nSame posterior, same data, same code. The metric chose which of those two rows "
          "you ship.")


_try("the wrong metric wins", _the_wrong_metric_wins, needs=_FOR_SWEEP + ("exercise 9",))

In [ ]:
def _plot_the_sweep() -> None:
    s = metric_sweep()
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    ax.plot(QUANTILES, s["rmse"], marker="o", ms=3, label="RMSE, hours (left)")
    ax.set_xlabel("quantile of the predictive RUL distribution you act on")
    ax.set_ylabel("RMSE, hours")
    twin_ax = ax.twinx()
    twin_ax.semilogy(QUANTILES, s["score"], marker="s", ms=3, color="firebrick",
                     label="C-MAPSS asymmetric score (right, log)")
    twin_ax.set_ylabel("mean asymmetric score, log scale")
    i_r = int(np.argmin(s["rmse"]))
    i_s = int(np.argmin(s["score"]))
    ax.axvline(QUANTILES[i_r], ls="--", lw=1.0, color="C0")
    ax.axvline(QUANTILES[i_s], ls="--", lw=1.0, color="firebrick")
    ax.set_title("the two metrics disagree about which model to ship")
    lines = ax.get_lines()[:1] + twin_ax.get_lines()[:1]
    ax.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="upper center")
    fig.tight_layout()
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)
    print(f"plotted: RMSE bottoms out at q={QUANTILES[i_r]:.2f}, the asymmetric score at "
          f"q={QUANTILES[i_s]:.2f}")


_try("the sweep, plotted", _plot_the_sweep, needs=_FOR_SWEEP)

## 12. Common mistakes

- **Reporting a number instead of a distribution.** "RUL 96 hours" is unactionable without
  its width: 96 ± 8 books the Thursday slot, 96 ± 190 books nothing. The whole of section 6
  exists to produce the second half of that sentence.
- **Fitting on the whole row.** Everything after the checkpoint is the future, and on this
  fleet the future contains a repair and a different machine. Exercise 4's check puts a
  nonsense tail on the row for exactly this reason.
- **Ignoring the prior at hour 96.** Twelve noisy readings will not identify a slope. The
  archive has a hundred units' worth of slopes and they are free.
- **Dropping the suspended units from the prior.** They never failed, so they have no
  end-of-life — and they still have a degradation rate, which is what the prior is over.
  Module 4 made this argument about labels; it is the same argument about parameters.
- **Letting `b <= 0` draws through.** A negative slope has no crossing time. Reported as a
  negative RUL it drags your lower quantile below zero and your alarm fires immediately.
- **Taking `log(health)` rather than `log(health - HEALTHY)`.** The first is not linear in
  time, and the slope you fit will depend on where in life you fitted it.
- **Ranking models on RMSE.** Section 11 is the whole answer to this one, on your numbers.
- **Reading a good alpha-lambda accuracy as safety.** The cone is symmetric. Two models
  with the same accuracy can fail in opposite directions, and only one of those directions
  leaves you with a broken machine.
- **Trusting an interval you have not checked.** A 90% predictive interval should contain
  the truth about 90% of the time. The cell below measures it on this fleet.

In [ ]:
def _check_calibration() -> None:
    """A predictive interval is a claim about frequencies. Measure whether it holds."""
    _, _, true, matrix = fleet_predictions()
    keep = true <= EVAL_HORIZON
    lo, hi = matrix[keep, 0], matrix[keep, -1]
    truth = true[keep]
    nominal = QUANTILES[-1] - QUANTILES[0]
    covered = float(np.mean((truth >= lo) & (truth <= hi)))
    below = float(np.mean(truth < lo))
    above = float(np.mean(truth > hi))
    print(f"the {nominal:.0%} predictive interval covers the truth {covered:.1%} of the time "
          f"on {int(keep.sum()):,} predictions")
    print(f"  truth below the interval  {below:>6.1%}   (the unit failed sooner than the "
          f"model's worst case)")
    print(f"  truth above the interval  {above:>6.1%}   (the unit outlived the model's best "
          f"case)")
    mid = QUANTILES.tolist().index(0.50)
    point = np.abs(matrix[keep, mid] - truth)
    print(f"\nThe median alone is out by {np.median(point):.0f} hours in the middle of its "
          f"own distribution.\nThe interval is what tells you that, and a point estimate "
          f"cannot carry it.")
    print(f"Coverage {covered:.1%} against a nominal {nominal:.0%} is the number to argue "
          f"about before\nanybody argues about the model.")


_try("is the interval honest", _check_calibration, needs=_FOR_TRACKS)

## 13. Self-check

1. Your model reports a median RUL of 96 hours for two units. On the first the 5%-95%
   interval is 88-104 hours; on the second it is 20-400. Acting identically on both is:
   - (a) correct, because the best estimate is the same
   - (b) irrelevant, since the interval is only presentation
   - (c) wrong, because the second unit's distribution supports no decision at the lead
         time you need, and the right action is another week of evidence, not a job card

2. Two RUL models are compared on a held-out fleet and their RMSEs differ by a fraction of
   an hour, with model B's the larger. The right conclusion is:
   - (a) nothing yet: RMSE cannot say which of them is late — section 11 printed the late
         rate, the asymmetric score and the short-warning count for exactly this pair shape
   - (b) Model A is better and should be deployed
   - (c) the difference is not significant, so pick either

3. A colleague proposes dropping the suspended units from the training set because "they
   never failed, so they have no label". What have they thrown away?
   - (a) nothing; without an end-of-life the unit teaches you nothing
   - (b) the degradation slopes of a fifth of the archive, which is what the prior over `b`
         is made of — censoring destroys the failure time, not the rate
   - (c) only some noise

4. Your alpha-lambda accuracy is 88% and your manager is pleased. The next number to put
   in front of them is:
   - (a) the RMSE, for a second opinion
   - (b) the coverage of the 90% interval, since that is the calibration question
   - (c) the split of the other 12% into early and late, because the cone is symmetric and
         those two failures cost different amounts

5. A prognostic that fires at a predicted 72 hours gives a median 64 hours of real warning,
   and the workshop needs 48. Moving to a later quantile would raise the RMSE-optimal
   ranking. The reason not to is:
   - (a) RMSE is a bad metric in general
   - (b) the distribution of warning times has a left tail, and the units in that tail are
         the ones that fail unplanned — a mean-square average of errors never sees them
   - (c) later quantiles are harder to compute

Answers, with the reasoning, are in this lesson's worked solution in the course repository.

In [ ]:
# The deliverable. A remaining-useful-life model that ships without its evaluation window,
# its action level and the direction of its errors is a number somebody will act on in the
# wrong direction. Print the note that goes with the model.
def _handover() -> None:
    s = metric_sweep()
    q_safe, _ = best_quantile(QUANTILES, s["pred"], s["truth"], mean_phm_score, "min")
    i = QUANTILES.tolist().index(round(q_safe, 2))
    prior_mean, _, noise_sd = prior()
    leads = s["leads"][:, i]
    print("RUL MODEL — ships with the model, or the model does not ship")
    print(f"  model              exponential degradation, log-linear, Bayesian fit against a "
          f"fleet prior")
    print(f"  prior              {len(HISTORY)} archived units, slope "
          f"{prior_mean[1]:.5f}/h, residual sd {noise_sd:.3f}")
    print(f"  failure level      health index {FAILURE_LEVEL:.1f}, from module 4's labelling "
          f"policy, not fitted")
    print(f"  answer             a distribution: {N_SAMPLES:,} draws per checkpoint, "
          f"re-run every {CHECK_EVERY} h")
    print(f"  act on             the {q_safe:.0%} quantile, chosen by the asymmetric score, "
          f"NOT by RMSE")
    print(f"  evaluated on       {s['rows']:,} predictions within {EVAL_HORIZON:.0f} h of "
          f"failure, {len(FLEET_FAILURES)} units")
    print(f"  accuracy           RMSE {s['rmse'][i]:.1f} h · in cone {s['inside'][i]:.1%} · "
          f"late {s['late'][i]:.1%} · early {1 - s['inside'][i] - s['late'][i]:.1%}")
    print(f"  warning            median {np.median(leads):.0f} h at an action level of "
          f"{ACTION_RUL:.0f} h; {int(s['short'][i])} of {len(FLEET_FAILURES)} units under the "
          f"{LEAD_HOURS:.0f} h contract")
    print(f"  re-derive whenever the failure level, the action level or the lead time moves. "
          f"All three are\n  in the model, and none of them is in the training data.")


_try("handover note", _handover, needs=_FOR_SWEEP + ("exercise 9",))

## What you built, and where it goes next

You fitted a degradation model, carried its uncertainty all the way through to a predictive
distribution over remaining life, and then watched four metrics disagree about what to do
with it. The disagreement was not a bug in any of them. RMSE and the alpha-lambda cone
answer "how far out is this number"; the asymmetric score and the warning lead answer "what
happens to the plant when it is out in this direction". Only the second pair is a
maintenance question.

Module 6 takes this distribution and does the thing this module deliberately did not: it
prices the intervention. Not "which quantile", but "which Thursday", with a workshop that
has finite capacity and a calendar that does not care about your posterior. The generator
and the run table above go forward unchanged.

The habit to carry forward: **when somebody shows you an RUL number, ask for its interval;
when somebody shows you a leaderboard, ask which direction the errors point.**

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_log_signal),
                              ("exercise 2", _check_fit_posterior),
                              ("exercise 3", _check_rul_samples),
                              ("exercise 4", _check_rul_track),
                              ("exercise 5", _check_rul_rmse),
                              ("exercise 6", _check_phm_score),
                              ("exercise 7", _check_cone_breakdown),
                              ("exercise 8", _check_warning_lead),
                              ("exercise 9", _check_best_quantile)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))